# Notebook 04 — XGBoost Baseline Model

**What this notebook builds:**
1. Trains an XGBoost classifier on the TF-IDF + metadata feature matrix
2. Applies class weights to correct for Credit Reporting's 66% dominance
3. Uses GPU acceleration (Colab T4) for fast training
4. Evaluates on the validation set: accuracy, macro F1, per-class metrics
5. Plots confusion matrix and top feature importances
6. Saves the trained model to Drive

**Why XGBoost first?**  
XGBoost on TF-IDF is our **classical baseline**. It:
- Trains in ~2 minutes vs hours for BERT fine-tuning
- Is interpretable via feature importance
- Sets a performance floor that BERT must exceed to justify the cost
- Handles sparse matrices natively (no dense conversion needed)

**Before running:** In Colab, go to `Runtime → Change runtime type → T4 GPU`. XGBoost with `device='cuda'` is ~10x faster than CPU on 140k × 30k sparse features.

## Cell 1 — Mount Drive & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

PROJECT_DIR = '/content/drive/MyDrive/nlp_project'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
FEATURE_DIR = os.path.join(PROJECT_DIR, 'features')
MODEL_DIR   = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'XGBoost version: {xgb.__version__}')

# Detect GPU
try:
    import subprocess
    gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode()
    DEVICE = 'cuda'
    print('GPU detected — using cuda')
except:
    DEVICE = 'cpu'
    print('No GPU found — using cpu (slower but works)')

## Cell 2 — Load Features, Labels, and Mappings

In [ ]:
print('Loading feature matrices ...')
X_train = sp.load_npz(os.path.join(FEATURE_DIR, 'X_train.npz'))
X_val   = sp.load_npz(os.path.join(FEATURE_DIR, 'X_val.npz'))
X_test  = sp.load_npz(os.path.join(FEATURE_DIR, 'X_test.npz'))

y_train = np.load(os.path.join(FEATURE_DIR, 'y_train.npy'))
y_val   = np.load(os.path.join(FEATURE_DIR, 'y_val.npy'))
y_test  = np.load(os.path.join(FEATURE_DIR, 'y_test.npy'))

with open(os.path.join(DATA_DIR, 'label_mapping.json')) as f:
    mapping  = json.load(f)
    id2label = {int(k): v for k, v in mapping['id2label'].items()}
    label2id = mapping['label2id']

with open(os.path.join(DATA_DIR, 'class_weights.json')) as f:
    class_weights = {int(k): float(v) for k, v in json.load(f).items()}

CLASS_NAMES = [id2label[i] for i in sorted(id2label)]
N_CLASSES   = len(CLASS_NAMES)

print(f'X_train : {X_train.shape}')
print(f'X_val   : {X_val.shape}')
print(f'Classes : {CLASS_NAMES}')
print(f'Class weights: {class_weights}')

## Cell 3 — Build Per-Sample Weights

XGBoost accepts `sample_weight` in `fit()` — a weight per training row, not per class.
We map each row's label to its class weight, so minority class rows are upweighted.

In [ ]:
sample_weights = np.array([class_weights[label] for label in y_train])

print('Sample weight stats:')
print(f'  min  : {sample_weights.min():.3f}  (most common class — Credit Reporting)')
print(f'  max  : {sample_weights.max():.3f}  (rarest class — Money Transfer)')
print(f'  mean : {sample_weights.mean():.3f}')

## Cell 4 — Train XGBoost

**History of what went wrong:**

| Run | Weights | Early stop | Result | Why |
|---|---|---|---|---|
| Run 1 | ✅ sample_weight | rounds=20 | Stopped at iter 2 | Weights → unweighted val loss rises → early stop fires instantly |
| Run 2 | ✅ sample_weight | rounds=50 | Stopped at iter 0 | Same conflict, stronger regularisation made it worse |
| Run 3 | ❌ no weights | ❌ no early stop | 500 trees, 18% acc | Catastrophic overfitting — memorised noise |

**Now fixed:** No sample_weight + early stopping are now fully compatible — the conflict only existed when both were present simultaneously. Early stopping now correctly watches unweighted val mlogloss, which aligns with our training objective.

| Parameter | Value | Reason |
|---|---|---|
| `n_estimators` | 3000 | High ceiling — early stopping will cut short |
| `learning_rate` | 0.05 | Moderate — needs ~400–800 trees to converge |
| `colsample_bytree` | 0.5 | 15k of 30k features per tree |
| `early_stopping_rounds` | 100 | Stop when val plateaus for 100 rounds |
| No `sample_weight` | — | Aligned with early stopping criterion |

In [ ]:
model = xgb.XGBClassifier(
    n_estimators          = 3000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.5,
    min_child_weight      = 5,
    objective             = 'multi:softprob',
    num_class             = N_CLASSES,
    eval_metric           = 'mlogloss',
    device                = DEVICE,
    tree_method           = 'hist',
    random_state          = 42,
    verbosity             = 1,
    early_stopping_rounds = 100,   # stops when val doesn't improve for 100 rounds
)

print('Training XGBoost (up to 3000 trees, early stopping at patience=100) ...')
model.fit(
    X_train, y_train,
    eval_set = [(X_val, y_val)],   # no sample_weight — no conflict with early stopping
    verbose  = 200,
)

print(f'\nBest iteration : {model.best_iteration}')
print(f'Best val loss  : {model.best_score:.4f}')

## Cell 5 — Evaluate on Validation Set

We evaluate on **val**, not test. Test is held out until Notebook 06 (final evaluation).

Primary metrics:
- **Accuracy** — overall correctness
- **Macro F1** — average F1 across all classes with equal weight per class (penalises poor performance on minority classes equally)

In [ ]:
y_pred_val = model.predict(X_val)

acc      = accuracy_score(y_val, y_pred_val)
macro_f1 = f1_score(y_val, y_pred_val, average='macro')

print(f'Validation Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print(f'Validation Macro F1 : {macro_f1:.4f}  ({macro_f1*100:.2f}%)')
print()
print('--- Per-class report ---')
print(classification_report(y_val, y_pred_val, target_names=CLASS_NAMES, digits=3))

## Cell 6 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_val, y_pred_val)

# Normalise by true class (rows sum to 1) so imbalance doesn't dominate colour
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual',    fontsize=12)
ax.set_title('XGBoost — Normalised Confusion Matrix (Validation Set)', fontsize=13)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'xgb_confusion_matrix.png'), dpi=120)
plt.show()
print('Saved confusion matrix.')

## Cell 7 — Feature Importance

XGBoost reports `gain`-based feature importance: how much each feature reduces loss across all splits it's used in.

We reconstruct feature names (TF-IDF terms + metadata) to make the plot interpretable.  
Top features should be domain-specific terms — this is another sanity check.

In [ ]:
# Reconstruct feature names in the same order as the hstacked matrix
tfidf_vec = joblib.load(os.path.join(FEATURE_DIR, 'tfidf_vectorizer.joblib'))
meta_enc  = joblib.load(os.path.join(FEATURE_DIR, 'meta_encoder.joblib'))
ohe       = meta_enc['ohe']

tfidf_names = list(tfidf_vec.get_feature_names_out())
ohe_names   = list(ohe.get_feature_names_out())
num_names   = ['is_older_american', 'is_servicemember', 'narrative_len']
all_names   = np.array(tfidf_names + ohe_names + num_names)

# Get gain-based importance
scores = model.get_booster().get_score(importance_type='gain')
# XGBoost names features as f0, f1, ...
importance = np.zeros(len(all_names))
for fname, score in scores.items():
    idx = int(fname[1:])   # strip leading 'f'
    if idx < len(all_names):
        importance[idx] = score

top_n  = 25
top_idx = importance.argsort()[-top_n:][::-1]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(all_names[top_idx][::-1], importance[top_idx][::-1], color='steelblue', edgecolor='white')
ax.set_title(f'XGBoost — Top {top_n} Features by Gain', fontsize=13)
ax.set_xlabel('Gain')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'xgb_feature_importance.png'), dpi=120)
plt.show()

print(f'Top 25 features:')
for i in top_idx[:25]:
    print(f'  {all_names[i]:<35}  gain: {importance[i]:.1f}')

## Cell 8 — Save Model

In [ ]:
model_path = os.path.join(MODEL_DIR, 'xgb_baseline.json')
model.save_model(model_path)
print(f'Saved model to: {model_path}')
print(f'Model size    : {os.path.getsize(model_path)/1e6:.1f} MB')

## Cell 9 — Summary (paste this output to Claude)

In [ ]:
print('='*60)
print('  NOTEBOOK 04 SUMMARY — paste this output to Claude')
print('='*60)
print(f'Device used              : {DEVICE}')
print(f'Best iteration           : {model.best_iteration}')
print(f'Best val mlogloss        : {model.best_score:.4f}')
print()
print(f'Validation Accuracy      : {acc:.4f}  ({acc*100:.2f}%)')
print(f'Validation Macro F1      : {macro_f1:.4f}  ({macro_f1*100:.2f}%)')
print()
print('Per-class F1 on val:')
f1_per = f1_score(y_val, y_pred_val, average=None)
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:<45}  F1: {f1_per[i]:.3f}')
print()
print('Top 10 features by gain:')
for i in top_idx[:10]:
    print(f'  {all_names[i]:<35}  {importance[i]:.1f}')
print('='*60)